# 02. Decorators in practice

**Tier:** Deep. **Why you'd use this:** Production code wraps *methods* as often as free functions, keeps call counts or config on a decorator *object*, freezes arguments with `partial`, and chooses between an unlimited memo and a size-capped LRU. The `@` line is the same sugar; the wrapper shape changes.

**After this lesson you can:** write a class with `__call__` as a decorator; decorate an instance method without dropping `self`; pick `cache` or `lru_cache(maxsize=…)`; bind fixed arguments with `functools.partial`.
About 35 minutes. **Uses:** [Decorators](01-decorators-lesson.ipynb), [Scope](../05-functions/03-scope-lesson.ipynb), [Methods](../11-classes/01-classes-lesson.ipynb)


## Quick review

1. What does `@logged` above `def greet` mean without the `@` spelling?
2. Why call `functools.wraps(func)` inside a decorator?

<details><summary>Answers</summary>

`greet = logged(greet)` after the definition. So logs, `help`, and some frameworks still show `greet`, not `wrapper`.
</details>


## The idea

A decorator is any callable that takes a function and returns a callable. A **class** can be that callable: `__init__` receives the function, `__call__` runs the wrapper, and instance attributes hold state without a `nonlocal` box. Decorating a **method** uses the same `@` sugar, but the wrapper must accept the instance as its first argument and forward it when it calls the original. `functools.partial` freezes some arguments now and returns a new callable. `functools.cache` remembers every distinct call forever; `lru_cache(maxsize=n)` remembers only the newest `n` keys and evicts the rest. Both caches require hashable arguments, so prefer tuples of primitives over lists.

<details><summary>Explain more</summary>

`@Counted` on `def greet` means `greet = Counted(greet)`. If `Counted` is a class, that assignment stores a `Counted` *instance* under the name `greet`. Later `greet("Ada")` runs `Counted.__call__`. A function decorator on a method stores a plain function on the class; looking up `g.hello` binds `self` through the usual descriptor protocol, then calls your wrapper. A class decorator on a method does *not* bind that way unless you also implement `__get__` — so for methods, start with a function wrapper that forwards `self`.
</details>

Trace of `@Counted` on a free function, then a call:

| step | what runs | result |
|---|---|---|
| 1 | `Counted(greet)` | `Counted` instance; `func` stored; `calls=0` |
| 2 | name `greet` rebound | callers get that instance |
| 3 | `greet("Ada")` | `__call__` bumps `calls`, then runs `func` |


## See it work

Count calls with a class decorator, log a method without dropping `self`, compare `cache` with a tiny `lru_cache`, and freeze a tax rate with `partial`.


In [1]:
import functools


class Counted:
    # 1. Class decorator: store the function and visible state.
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)


@Counted
def greet(name: str) -> str:
    return "hi " + name


print(greet("Ada"), greet.calls)


def as_method(func):
    # 2. Method wrapper: first arg is the instance.
    @functools.wraps(func)
    def wrapper(self, name: str) -> str:
        print("call", func.__name__, self.label, name)
        return func(self, name)

    return wrapper


class Greeter:
    def __init__(self, label: str) -> None:
        self.label = label

    @as_method
    def hello(self, name: str) -> str:
        return self.label + " " + name


g = Greeter("hi")
print(g.hello("Bea"))


@functools.cache
def fib(n: int) -> int:
    # 3. Unlimited memo — fine for a small key space.
    return n if n < 2 else fib(n - 1) + fib(n - 2)


@functools.lru_cache(maxsize=2)
def recent(n: int) -> int:
    # 4. Tiny LRU — older keys fall out.
    return n * 10


print(fib(6), recent.cache_info())
print(recent(1), recent(2), recent(3), recent.cache_info())


def price(cents: int, rate: float) -> float:
    return cents * rate


# 5. Freeze one argument; callers pass the rest.
taxed = functools.partial(price, rate=1.1)
print(taxed(100), taxed(0))

hi Ada 1
call hello hi Bea
hi Bea
8 CacheInfo(hits=0, misses=0, maxsize=2, currsize=0)
10 20 30 CacheInfo(hits=0, misses=3, maxsize=2, currsize=2)
110.00000000000001 0.0


## Check yourself

1. After `@Counted` on `greet`, what is `greet.calls` counting?
2. Why must `as_method`'s wrapper take `self` before `name`?
3. What does `maxsize=2` change compared with `cache`?

<details><summary>Check your answers</summary>

How many times the wrapped callable ran. Instance lookup passes the `Greeter` as the first argument; the wrapper must accept and forward it. Only two argument keys stay in memory; older ones are evicted.
</details>


## Why it works this way

`f = deco(f)` does not care whether `deco` is a function or a class instance with `__call__`. After `@Counted` on a free function, the name holds a `Counted` *object*. Calling that name runs `__call__`, which is why `greet.calls` is ordinary attribute access on the decorator instance. A class keeps state (`calls`, config, a logger handle) on `self` without a `nonlocal` box and without wrapping state in a list. `update_wrapper` / `wraps` still matter: they copy `__name__` and related metadata onto the wrapper object so `help` and traces name the original function.

Methods are different. A function stored on a class is a descriptor: looking up `g.hello` builds a bound method that inserts `g` as the first argument. Your wrapper must accept that instance and pass it through when it calls the original. A *class* decorator on a method stores a non-function object on the class, so the usual binding does not run unless you also implement `__get__`. That is why this lesson decorates methods with a function wrapper and reserves class decorators for free functions and other plain callables.

`cache` and `lru_cache` hash the argument tuple. Lists and other unhashables fail until you convert them (often to a tuple of primitives). `cache` is `lru_cache` with no max size: every new key stays forever, which is fine for a tiny Fibonacci-sized space and risky in a long-lived process. `lru_cache(maxsize=n)` evicts the least-recently used key when full, so memory stays bounded and some calls recompute. `partial` stores the frozen args and merges them on each call; it does not memoize. Deep dive: [Scopes and closures](../../internals/06-scopes/01-closures-lesson.ipynb).

Cost: one extra Python call (or `__call__`) per decoration layer, plus whatever work you add. An unlimited cache grows with every distinct key. An LRU bounds that growth but pays for bookkeeping and occasional recomputation. Decorating many methods multiplies wrapper frames across the class, so keep wrappers thin.

Design reason: put reusable enhancement next to `def`, keep state where you can inspect it in tests, and choose a cache policy that matches how large and how long-lived the key space is. Prefer a nested function when it reads more clearly than `partial`, and prefer a function decorator when a class would only hold a single `func` field.

Evidence: `greet.calls` increments on each call, `g.hello("Bea")` prints then returns with `self` intact, `recent.cache_info()` shows eviction at `maxsize=2`, and `taxed(100)` applies the frozen rate without rewriting `price`.

Link: [Decorators](01-decorators-lesson.ipynb) for the `@` sugar; [Methods](../11-classes/01-classes-lesson.ipynb) for binding.


## When to use it

Use a **class decorator** when the wrapper needs visible state or configuration beyond a simple closure — call counts, feature flags, or a logger handle you want to inspect in tests. Decorate **methods** with a function wrapper that forwards `self`, and test through an instance so binding stays in the path. Prefer **`cache`** for pure functions on a small argument space (Fibonacci-sized); prefer **`lru_cache(maxsize=…)`** when the key space is large or the process is long-lived. Use **`partial`** to specialize a multi-argument function for callbacks and APIs that expect a narrower callable.

Skip class decorators for a one-line timer — a function decorator is enough. Skip caching when arguments are unhashable or results must stay fresh. Skip `partial` when a small nested function reads more clearly to the next reader.


## Common mistake

Decorating a method with a wrapper whose signature forgets that the instance arrives as the first argument.


In [2]:
def broken(func):
    def wrapper(name: str) -> str:
        return func(name)

    return wrapper


class BrokenGreeter:
    def __init__(self, label: str) -> None:
        self.label = label

    @broken
    def hello(self, name: str) -> str:
        return self.label + " " + name


BrokenGreeter("hi").hello("Ada")

TypeError: broken.<locals>.wrapper() takes 1 positional argument but 2 were given

The call raises `TypeError` because `wrapper` expected only `name`, but the bound-method call passed the instance first. The fix forwards `self`:

```python
def wrapper(self, name: str) -> str:
    return func(self, name)
```


In [3]:
def fixed(func):
    def wrapper(self, name: str) -> str:
        return func(self, name)

    return wrapper


class FixedGreeter:
    def __init__(self, label: str) -> None:
        self.label = label

    @fixed
    def hello(self, name: str) -> str:
        return self.label + " " + name


print(FixedGreeter("hi").hello("Ada"))

hi Ada


## Try it

Change `Counted` so it also stores the last arguments on `self.last_args`, then print them after two calls to `greet`.


## Recap

- A class with `__call__` can be a decorator and hold state.
- Method wrappers must forward `self`.
- `cache` keeps everything; `lru_cache(maxsize=n)` bounds memory.
- `partial` freezes arguments without caching.

**Next:** [Practice](02-decorators-in-practice-practice.ipynb)
